# 03 · Fault test and API timing (R9, R11, Week 8 Milestone 1 C and D)
Plays the same messy hour through the API **with and without the edge range check**, then times the two
ways of serving the model (Week 8 designs C and D). Everything runs in this notebook: no Docker needed.

In [1]:
# SETUP: make the project's own code importable, whatever folder VS Code started in.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 30)
print("Project root:", ROOT)

Project root: /Users/admin/Desktop/smart-vitals-ipms


In [2]:
import json, time
from simulator.generator import simulate
from simulator import edge
from tests.helpers import make_client, post
from src import config

register = json.loads(config.PATIENTS_PATH.read_text())
events, observed = simulate(register, minutes=45, seed=11, faults=True)
truth = {o["reading_id"]: o["true_label"] for o in observed}
print(len(events), "messages")

862 messages


## Edge ON vs edge OFF

In [3]:
def play(use_edge):
    client, rows = make_client(register), []
    for ev in events:
        reading = edge.range_check(ev["reading"])[0] if use_edge else ev["reading"]
        r = post(client, reading)
        rows.append({"fault": ev["fault"] or "clean", "code": r.status_code,
                     "outcome": r.get_json().get("outcome"), "alert": bool(r.get_json().get("alert"))})
    return pd.DataFrame(rows)

with_edge, without_edge = play(True), play(False)
pd.DataFrame({"edge ON": with_edge.groupby(["code", "outcome"]).size(),
              "edge OFF": without_edge.groupby(["code", "outcome"]).size()}).fillna(0).astype(int)

,,edge ON,edge OFF
code,outcome,,
200,duplicate,30,29
201,created,832,820
422,invalid,0,13


**Milestone 1 C, what did validation throw away?** With the edge OFF, every glitched reading is refused with a 422, so its *good* vitals are lost too.
With the edge ON, only the impossible value is blanked and the rest of the reading is kept.

In [4]:
g_off = without_edge[without_edge.fault == "glitch"]
g_on = with_edge[with_edge.fault == "glitch"]
print(f"glitched readings: {len(g_off)}")
print(f"  edge OFF: {int((g_off.code == 422).sum())} refused (whole reading lost)")
print(f"  edge ON : {int((g_on.code == 201).sum())} stored with the bad value blanked")

glitched readings: 10
  edge OFF: 10 refused (whole reading lost)
  edge ON : 10 stored with the bad value blanked


**YOUR TURN:** which is better for the patient, and what is the cost of doing the check at the edge? (Week 4 trade-offs; R9.)

## Design C vs D: load the model once, or on every request (Week 8)

In [5]:
import joblib
from app.classify import classify
from src import model as M
from tests.helpers import normal_reading

patient = {"patient_id": "P001", "risk_group": "Medium", "has_diabetes": 1}
reading = normal_reading()
model = M.load_model()

def time_it(fn, n=40):
    out = []
    for _ in range(n):
        t = time.perf_counter(); fn(); out.append((time.perf_counter() - t) * 1000)
    return np.percentile(out, [50, 95]).round(1)

c_times = time_it(lambda: classify(model, patient, [], reading))
d_times = time_it(lambda: classify(joblib.load(config.MODEL_PATH), patient, [], reading))
pd.DataFrame({"p50 (ms)": [c_times[0], d_times[0]], "p95 (ms)": [c_times[1], d_times[1]]},
             index=["C: model loaded once", "D: model loaded every request"])

,p50 (ms),p95 (ms)
C: model loaded once,52.1,55.3
D: model loaded every request,134.3,148.9


**YOUR TURN:** write ADR 3 using these numbers (your times will differ: say what machine you used).